In [13]:
import pandas as pd

test = pd.read_csv('cognicart/data/cleaned/supermarket_clean.csv')
print(f"Total rows: {len(test)}")
print(f"Total transactions: {test['TransactionID'].nunique()}")
print(f"Avg items per transaction: {test.groupby('TransactionID')['Product'].count().mean():.2f}")
print(test.head(10))

Total rows: 27596
Total transactions: 5000
Avg items per transaction: 5.52
   TransactionID Customer_ID        Date DayOfWeek     Month  \
0              1       C0027  2024-02-17  Saturday  February   
1              1       C0027  2024-02-17  Saturday  February   
2              1       C0027  2024-02-17  Saturday  February   
3              1       C0027  2024-02-17  Saturday  February   
4              1       C0027  2024-02-17  Saturday  February   
5              1       C0027  2024-02-17  Saturday  February   
6              1       C0027  2024-02-17  Saturday  February   
7              1       C0027  2024-02-17  Saturday  February   
8              2       C0108  2024-12-15    Sunday  December   
9              2       C0108  2024-12-15    Sunday  December   

             Category      Product  Quantity   Price    CustomerType  \
0  Fruit & Vegetables       Banana         4  280.32  student_budget   
1           Beverages   Cold Drink         4  284.84  student_budget   
2   

In [1]:
!pip install sentence-transformers

Defaulting to user installation because normal site-packages is not writeable


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import KMeans
import warnings
warnings.filterwarnings('ignore')

print('All libraries imported!')


All libraries imported!


In [3]:
print('Loading BERT model... please wait')
bert_model = SentenceTransformer('all-MiniLM-L6-v2')
print('BERT model loaded!')


Loading BERT model... please wait


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BERT model loaded!


In [4]:

df = pd.read_csv('cognicart/data/cleaned/supermarket_clean.csv')
products = df['Product'].unique().tolist()
products.sort()

print(f'Total products: {len(products)}')
print('Products list:')
for i, p in enumerate(products):
    print(f'  {i+1:2d}. {p}')


Total products: 43
Products list:
   1. Apple
   2. Banana
   3. Biscuit
   4. Biscuits
   5. Bread
   6. Brown Bread
   7. Butter
   8. Carrot
   9. Cheese
  10. Chips
  11. Chocolate
  12. Coffee
  13. Cold Drink
  14. Cookies
  15. Curd
  16. Dal
  17. Facewash
  18. Ghee
  19. Juice
  20. Milk
  21. Namkeen
  22. Nuts
  23. Oats
  24. Oil
  25. Onion
  26. Panner
  27. Poha
  28. Popcorn
  29. Potato
  30. Rice
  31. Rusk
  32. Salt
  33. Semolina
  34. Shampoo
  35. Soap
  36. Spices
  37. Sugar
  38. Tea
  39. Tomato
  40. Tomatosauce
  41. Toothpaste
  42. Waterbottle
  43. Wheat Flour


In [5]:
# Generate BERT embeddings for all products
print('Generating embeddings...')
embeddings = bert_model.encode(products, show_progress_bar=True)

print(f'Embeddings shape: {embeddings.shape}')
print(f'Each product is now a vector of {embeddings.shape[1]} numbers')
print()
print('Example - first product embedding (first 10 numbers):')
print(f'{products[0]} → {embeddings[0][:10].round(4)}')


Generating embeddings...


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Embeddings shape: (43, 384)
Each product is now a vector of 384 numbers

Example - first product embedding (first 10 numbers):
Apple → [-0.0061  0.031   0.0648  0.0109  0.0053 -0.0475  0.0812  0.029   0.0668
  0.0303]


In [6]:
# Calculate similarity between all products
similarity_matrix = cosine_similarity(embeddings)

# Create similarity DataFrame
similarity_df = pd.DataFrame(
    similarity_matrix,
    index=products,
    columns=products
)

print('Similarity matrix created!')
print(f'Shape: {similarity_df.shape}')
print()

# Show most similar products for each product
def get_similar_products(product_name, top_n=5):
    if product_name not in similarity_df.index:
        print(f'Product not found: {product_name}')
        return
    similarities = similarity_df[product_name].sort_values(ascending=False)
    similar = similarities[similarities.index != product_name].head(top_n)
    print(f'Products similar to: {product_name}')
    for prod, score in similar.items():
        print(f'  {prod:20s}  Similarity: {score:.3f}')
    print()

# Test with different products
get_similar_products('Milk')
get_similar_products('Rice')
get_similar_products('Bread')


Similarity matrix created!
Shape: (43, 43)

Products similar to: Milk
  Cheese                Similarity: 0.628
  Chocolate             Similarity: 0.605
  Butter                Similarity: 0.568
  Sugar                 Similarity: 0.553
  Coffee                Similarity: 0.540

Products similar to: Rice
  Oats                  Similarity: 0.522
  Potato                Similarity: 0.513
  Banana                Similarity: 0.499
  Wheat Flour           Similarity: 0.486
  Milk                  Similarity: 0.477

Products similar to: Bread
  Brown Bread           Similarity: 0.782
  Biscuits              Similarity: 0.586
  Wheat Flour           Similarity: 0.570
  Cheese                Similarity: 0.538
  Biscuit               Similarity: 0.516



In [7]:
# Calculate similarity between all products
similarity_matrix = cosine_similarity(embeddings)

# Create similarity DataFrame
similarity_df = pd.DataFrame(
    similarity_matrix,
    index=products,
    columns=products
)

print('Similarity matrix created!')
print(f'Shape: {similarity_df.shape}')
print()

# Show most similar products for each product
def get_similar_products(product_name, top_n=5):
    if product_name not in similarity_df.index:
        print(f'Product not found: {product_name}')
        return
    similarities = similarity_df[product_name].sort_values(ascending=False)
    similar = similarities[similarities.index != product_name].head(top_n)
    print(f'Products similar to: {product_name}')
    for prod, score in similar.items():
        print(f'  {prod:20s}  Similarity: {score:.3f}')
    print()

# Test with different products
get_similar_products('Milk')
get_similar_products('Rice')
get_similar_products('Bread')


Similarity matrix created!
Shape: (43, 43)

Products similar to: Milk
  Cheese                Similarity: 0.628
  Chocolate             Similarity: 0.605
  Butter                Similarity: 0.568
  Sugar                 Similarity: 0.553
  Coffee                Similarity: 0.540

Products similar to: Rice
  Oats                  Similarity: 0.522
  Potato                Similarity: 0.513
  Banana                Similarity: 0.499
  Wheat Flour           Similarity: 0.486
  Milk                  Similarity: 0.477

Products similar to: Bread
  Brown Bread           Similarity: 0.782
  Biscuits              Similarity: 0.586
  Wheat Flour           Similarity: 0.570
  Cheese                Similarity: 0.538
  Biscuit               Similarity: 0.516



In [8]:
# Group similar products into semantic clusters
NUM_CLUSTERS = 8  # One per category ideally

kmeans = KMeans(n_clusters=NUM_CLUSTERS, random_state=42, n_init=10)
product_clusters = kmeans.fit_predict(embeddings)

# Create product cluster mapping
product_cluster_df = pd.DataFrame({
'Product': products,
'SemanticCluster': product_clusters
})

# Show products in each cluster
print('Semantic Product Clusters (found by BERT):')
print('=' * 45)
for cluster in range(NUM_CLUSTERS):
    cluster_products = product_cluster_df[
        product_cluster_df['SemanticCluster'] == cluster
    ]['Product'].tolist()
    print(f'Cluster {cluster}: {cluster_products}')
    print()


Semantic Product Clusters (found by BERT):
Cluster 0: ['Apple', 'Banana', 'Carrot', 'Namkeen', 'Nuts', 'Potato', 'Rice', 'Tomato', 'Tomatosauce']

Cluster 1: ['Biscuit', 'Biscuits', 'Bread', 'Brown Bread', 'Chips', 'Cookies', 'Oats', 'Wheat Flour']

Cluster 2: ['Dal', 'Oil', 'Onion', 'Salt', 'Spices']

Cluster 3: ['Facewash', 'Shampoo', 'Soap']

Cluster 4: ['Coffee', 'Cold Drink', 'Juice', 'Tea', 'Waterbottle']

Cluster 5: ['Butter', 'Cheese', 'Chocolate', 'Curd', 'Milk', 'Panner', 'Popcorn', 'Sugar', 'Toothpaste']

Cluster 6: ['Ghee', 'Poha']

Cluster 7: ['Rusk', 'Semolina']



In [9]:
pip install mlxtend

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [10]:
from mlxtend.frequent_patterns import fpgrowth, association_rules
from mlxtend.preprocessing import TransactionEncoder

df = pd.read_csv('cognicart/data/cleaned/supermarket_clean.csv')
df_bert = df.merge(product_cluster_df, on='Product', how='left')

basket_semantic = df_bert.groupby('TransactionID').agg(
    Products=('Product', list),
    Clusters=('SemanticCluster', list)
).reset_index()

te = TransactionEncoder()
basket_list = df.groupby('TransactionID')['Product'].apply(list).tolist()
te_array = te.fit_transform(basket_list)
basket_df = pd.DataFrame(te_array, columns=te.columns_)

frequent_items = fpgrowth(basket_df, min_support=0.02, use_colnames=True)
print(f"Frequent itemsets found: {len(frequent_items)}")

rules = association_rules(frequent_items, metric='lift', min_threshold=1.0)
print(f"Rules found: {len(rules)}")

def get_semantic_similarity(antecedent, consequent):
    ant_list = list(antecedent)
    con_list = list(consequent)
    if ant_list and con_list:
        ant_emb = bert_model.encode(ant_list[0])
        con_emb = bert_model.encode(con_list[0])
        sim = cosine_similarity([ant_emb], [con_emb])[0][0]
        return round(float(sim), 3)
    return 0

rules['SemanticSimilarity'] = rules.apply(
    lambda row: float(get_semantic_similarity(row['antecedents'], row['consequents'])),
    axis=1
)

print(f'Total rules with semantic similarity: {len(rules)}')
print(rules[['antecedents','consequents','confidence','lift','SemanticSimilarity']].head(10))

Frequent itemsets found: 305
Rules found: 1020
Total rules with semantic similarity: 1020
                antecedents               consequents  confidence      lift  \
0          frozenset({Tea})        frozenset({Bread})    0.612291  1.550105   
1        frozenset({Bread})          frozenset({Tea})    0.408608  1.550105   
2          frozenset({Tea})         frozenset({Milk})    0.408194  1.105618   
3         frozenset({Milk})          frozenset({Tea})    0.291441  1.105618   
4   frozenset({Tea, Bread})         frozenset({Milk})    0.443618  1.201566   
5    frozenset({Tea, Milk})        frozenset({Bread})    0.665428  1.684627   
6  frozenset({Milk, Bread})          frozenset({Tea})    0.402700  1.527692   
7          frozenset({Tea})  frozenset({Milk, Bread})    0.271624  1.527692   
8        frozenset({Bread})    frozenset({Tea, Milk})    0.181266  1.684627   
9         frozenset({Milk})   frozenset({Tea, Bread})    0.193933  1.201566   

   SemanticSimilarity  
0               

In [12]:
import pickle
import numpy as np
import os

os.makedirs('cognicart/models', exist_ok=True)
os.makedirs('cognicart/data/cleaned', exist_ok=True)

np.save('cognicart/models/product_embeddings.npy', embeddings)
product_cluster_df.to_csv('cognicart/data/cleaned/product_clusters.csv', index=False)
similarity_df.to_csv('cognicart/data/cleaned/product_similarity.csv')

rules['antecedents'] = rules['antecedents'].astype(str)
rules['consequents'] = rules['consequents'].astype(str)
rules.to_csv('cognicart/data/cleaned/enhanced_rules.csv', index=False)

print('All Week 6 files saved!')
print('  cognicart/models/product_embeddings.npy')
print('  cognicart/data/cleaned/product_clusters.csv')
print('  cognicart/data/cleaned/product_similarity.csv')
print('  cognicart/data/cleaned/enhanced_rules.csv')

All Week 6 files saved!
  cognicart/models/product_embeddings.npy
  cognicart/data/cleaned/product_clusters.csv
  cognicart/data/cleaned/product_similarity.csv
  cognicart/data/cleaned/enhanced_rules.csv


In [11]:
print('=' * 50)
print('WEEK 6 COMPLETE!')
print('=' * 50)
print('What you built:')
print('  Loaded Sentence-BERT model')
print('  Generated 384-dim embeddings for all products')
print('  Calculated cosine similarity between products')
print('  Clustered similar products semantically')
print('  Enhanced association rules with similarity scores')
print()


WEEK 6 COMPLETE!
What you built:
  Loaded Sentence-BERT model
  Generated 384-dim embeddings for all products
  Calculated cosine similarity between products
  Clustered similar products semantically
  Enhanced association rules with similarity scores

